# 02 - Data Analysis
This notebook focuses on using the processed fitness dataset to discover and explore trends in strength progression. This will be done by calculating an estimated one rep max for key exercises to have a standardized measure of strength, then using plots to visualize these trends in conjunction with related data such as workout frequency and bodyweight.

In [148]:
import pandas as pd
import numpy as np
import plotly.express as px

pd.set_option('display.max_columns', None)  # display all columns
pd.set_option('display.max_rows', 150)  # display up to 150 rows
pd.set_option('display.max_colwidth', None)  # increase column width
pd.set_option('display.float_format', '{:.2f}'.format)  # format floats to 2 decimal places 

In [149]:
processed_df = pd.read_csv('../data/processed/processed_data.csv')
processed_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2688 entries, 0 to 2687
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   date                2688 non-null   object 
 1   bodyweight_lbs      2571 non-null   float64
 2   workout_title       2042 non-null   object 
 3   start_time          2042 non-null   object 
 4   end_time            2042 non-null   object 
 5   exercise_title      2042 non-null   object 
 6   set_index           2042 non-null   float64
 7   set_type            2042 non-null   object 
 8   working_weight_lbs  1957 non-null   float64
 9   reps                2042 non-null   float64
 10  rpe                 2004 non-null   float64
dtypes: float64(5), object(6)
memory usage: 231.1+ KB


### 2.1 - Determining Estimated One Rep Max
An estimated one rep max will be used as a standard measure of strength. This number will be obtained by using a lookup dictionary that gives a percentage of maximum strength based on reps and RPE for the set. These percentages are based on the findings of Mike Tuchscherer of Reactive Training Systems. This is then used in the `calculate_e1rm` function, after which a new column with the value is added to the processed data DataFrame.

In [150]:
rpe_chart = {
    10: {1: 1.0, 2: 0.955, 3: 0.922, 4: 0.892, 5: 0.863, 6: 0.837, 7: 0.811, 8: 0.786, 9: 0.762, 10: 0.739, 11: 0.707, 12: 0.68, 13: 0.653, 14: 0.626, 15: 0.599},
    9.5: {1: 0.9775, 2: 0.9385, 3: 0.907, 4: 0.8775, 5: 0.85, 6: 0.824, 7: 0.7985, 8: 0.774, 9: 0.7505, 10: 0.723, 11: 0.6935, 12: 0.6665, 13: 0.6395, 14: 0.6125, 15: 0.5855},
    9: {1: 0.955, 2: 0.922, 3: 0.892, 4: 0.863, 5: 0.837, 6: 0.811, 7: 0.786, 8: 0.762, 9: 0.739, 10: 0.707, 11: 0.68, 12: 0.653, 13: 0.626, 14: 0.599, 15: 0.572},
    8.5: {1: 0.9385, 2: 0.907, 3: 0.8775, 4: 0.85, 5: 0.824, 6: 0.7985, 7: 0.774, 8: 0.7505, 9: 0.723, 10: 0.6935, 11: 0.6665, 12: 0.6395, 13: 0.6125, 14: 0.5855, 15: 0.5585},
    8: {1: 0.922, 2: 0.892, 3: 0.863, 4: 0.837, 5: 0.811, 6: 0.786, 7: 0.762, 8: 0.739, 9: 0.707, 10: 0.68, 11: 0.653, 12: 0.626, 13: 0.599, 14: 0.572, 15: 0.545},
    7.5: {1: 0.907, 2: 0.8775, 3: 0.85, 4: 0.824, 5: 0.7985, 6: 0.774, 7: 0.7505, 8: 0.723, 9: 0.6935, 10: 0.6665, 11: 0.6395, 12: 0.6125, 13: 0.5855, 14: 0.5585, 15: 0.5315},
    7: {1: 0.892, 2: 0.863, 3: 0.837, 4: 0.811, 5: 0.786, 6: 0.762, 7: 0.739, 8: 0.707, 9: 0.68, 10: 0.653, 11: 0.626, 12: 0.599, 13: 0.572, 14: 0.545, 15: 0.518},
    6: {1: 0.863, 2: 0.837, 3: 0.811, 4: 0.786, 5: 0.762, 6: 0.739, 7: 0.707, 8: 0.68, 9: 0.653, 10: 0.626, 11: 0.599, 12: 0.572, 13: 0.545, 14: 0.518, 15: 0.491}
}

In [151]:
def calculate_e1rm(row):
    try:
        # obtain the information about the set necessary for calculating an estimated one rep max
        working_weight = row['working_weight_lbs']
        reps = row['reps']
        rpe = row['rpe']

        # disregard sets with over 15 reps since they are not accounted for in the chart and are irrelevant for determining strength
        if reps > 15:
            return None

        # return the estimated one rep max rounded to 1 decimal point
        percentage = rpe_chart[rpe][reps]
        return round(working_weight / percentage, 1)
    
    # return nothing when values are missing
    except (KeyError, TypeError):
        return None

In [152]:
processed_df['estimated_1rm'] = processed_df.apply(calculate_e1rm, axis=1)
processed_df.info()  # for verifying the data was added properly

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2688 entries, 0 to 2687
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   date                2688 non-null   object 
 1   bodyweight_lbs      2571 non-null   float64
 2   workout_title       2042 non-null   object 
 3   start_time          2042 non-null   object 
 4   end_time            2042 non-null   object 
 5   exercise_title      2042 non-null   object 
 6   set_index           2042 non-null   float64
 7   set_type            2042 non-null   object 
 8   working_weight_lbs  1957 non-null   float64
 9   reps                2042 non-null   float64
 10  rpe                 2004 non-null   float64
 11  estimated_1rm       1885 non-null   float64
dtypes: float64(6), object(6)
memory usage: 252.1+ KB


### 2.2 - Analyzing Bodyweight Trends
There are three underlying phases within the bodyweight data:
1. **Before Cancer Diagnosis:** During this period I was working out regularly, but without recording them. 
2. **During Cancer Treatment:** This period understandably caused a significant change in lifestyle where I engaged in very little activity.
3. **After Cancer Treatment:** Almost immediately after my final chemotherapy infusion I began working out again, and started tracking them as well. Therefore, all the workout data comes from this period.

Using vertical lines to mark these events, we can analyze how the weight changes differed within each phase.

In [153]:
cancer_diagnosis = pd.to_datetime('02/15/2024')  # convert the date to a pd datetime object for analysis
first_recorded_workout = processed_df[processed_df['exercise_title'].notna()]['date'].min()  # find the first recorded workout date

In [154]:
bodyweight_chart = px.line(processed_df, 'date', 'bodyweight_lbs', labels={'date': 'Date', 'bodyweight_lbs': 'Bodyweight (lbs)'})
bodyweight_chart.add_vline(cancer_diagnosis)
bodyweight_chart.add_vline(first_recorded_workout)
bodyweight_chart.show()